# Esplorazione del Modello SIR come Catena di Markov

**Corso**: Modelli Probabilistici — Università di Bologna  
**Docente**: Prof. Salvatore Federico  
**Studente**: Francesco Castaldi  
**A.A.**: 2025/2026

---

Questo notebook permette di esplorare interattivamente il modello SIR interpretato
come catena di Markov a tempo discreto. Modificando i parametri si osserva l'effetto
sulla dinamica epidemica.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "..")

from src.model import next_state, N, BETA, GAMMA, I0, T_MAX, M
from src.simulation import run_single, run_montecarlo
from src.analysis import extinction_time, compute_stats

print("Moduli importati con successo.")

## 1. Parametri del Modello

Modifica i valori nelle celle sottostanti per esplorare scenari diversi.

In [ ]:
# Parametri modificabili
POPULATION = N           # 100
BETA = BETA              # 0.2 — probabilità di contagio per contatto
GAMMA = GAMMA            # 0.1 — probabilità di guarigione
INITIAL_INFECTED = I0    # 5
T_MAX = T_MAX            # 200 — orizzonte temporale
N_SIMULATIONS = 1000     # Numero di repliche Monte Carlo

print(f"N = {POPULATION}, β = {BETA}, γ = {GAMMA}")
print(f"I₀ = {INITIAL_INFECTED}, T_MAX = {T_MAX}, M = {N_SIMULATIONS}")

## 2. Singola Traiettoria

Eseguiamo una singola simulazione e osserviamo l'evoluzione stocastica.

In [ ]:
traj = run_single(n=POPULATION, i0=INITIAL_INFECTED,
                  t_max=T_MAX, beta=BETA, gamma=GAMMA)

plt.figure(figsize=(10, 4))
steps = range(len(traj))
plt.plot(steps, traj[:, 0], label="S (Suscettibili)", color="steelblue")
plt.plot(steps, traj[:, 1], label="I (Infetti)", color="firebrick")
plt.plot(steps, traj[:, 2], label="R (Rimossi)", color="seagreen")
plt.xlabel("Passo temporale $t$")
plt.ylabel("Individui")
plt.title(f"Singola traiettoria SIR (β={BETA}, γ={GAMMA})")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Tempo di estinzione: τ = {extinction_time(traj)}")
print(f"Picco infetti: I_max = {np.max(traj[:, 1]):.0f}")
print(f"Rimossi finali: R_∞ = {traj[-1, 2]:.0f}")

## 3. Simulazione Monte Carlo

Eseguiamo M repliche per studiare il comportamento medio e la variabilità.

In [ ]:
results = run_montecarlo(m=N_SIMULATIONS, n=POPULATION, i0=INITIAL_INFECTED,
                         t_max=T_MAX, beta=BETA, gamma=GAMMA)
stats = compute_stats(results)

print(f"=== Statistiche su {len(results)} simulazioni ===")
for k, v in stats.items():
    print(f"  {k}: {v:.2f}")

## 4. Traiettoria Media

Media e deviazione standard delle tre componenti su tutte le simulazioni.

In [ ]:
def pad_results(results):
    max_len = max(len(r) for r in results)
    def pad_col(col):
        mat = np.full((len(results), max_len), np.nan)
        for k, traj in enumerate(results):
            mat[k, :len(traj)] = traj[:, col]
        return mat
    return pad_col(0), pad_col(1), pad_col(2), max_len

s_mat, i_mat, r_mat, max_len = pad_results(results)
steps = range(max_len)

plt.figure(figsize=(10, 4))
for mat, color, label in [
    (s_mat, "steelblue", "S"),
    (i_mat, "firebrick", "I"),
    (r_mat, "seagreen", "R"),
]:
    mean = np.nanmean(mat, axis=0)
    std = np.nanstd(mat, axis=0)
    plt.plot(steps, mean, color=color, label=f"E[{label}]")
    plt.fill_between(steps, mean - std, mean + std, color=color, alpha=0.15)

plt.xlabel("Passo temporale $t$")
plt.ylabel("Individui")
plt.title(f"Traiettorie medie ± 1 std — {len(results)} simulazioni")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Distribuzione del Tempo di Estinzione

Istogramma del tempo τ in cui I raggiunge 0 per la prima volta.

In [ ]:
taus = [extinction_time(r) for r in results]

plt.figure(figsize=(9, 4))
plt.hist(taus, bins=30, color="slateblue", edgecolor="white")
plt.axvline(np.mean(taus), color="darkred", linestyle="--",
            label=f"Media τ = {np.mean(taus):.1f}")
plt.xlabel("Tempo di estinzione $\\tau$")
plt.ylabel("Frequenza")
plt.title("Distribuzione empirica del tempo di estinzione")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Effetto dei Parametri

Confrontiamo l'effetto di β e γ sul picco e sulla durata dell'epidemia.

In [ ]:
scenarios = [
    ("β=0.15, γ=0.1", 0.15, 0.1),
    ("β=0.20, γ=0.1", 0.20, 0.1),
    ("β=0.25, γ=0.1", 0.25, 0.1),
    ("β=0.20, γ=0.05", 0.20, 0.05),
    ("β=0.20, γ=0.15", 0.20, 0.15),
]

plt.figure(figsize=(12, 5))
colors = ["steelblue", "firebrick", "seagreen", "darkorange", "purple"]

for (label, beta, gamma), color in zip(scenarios, colors):
    res = run_montecarlo(m=200, n=POPULATION, i0=INITIAL_INFECTED,
                         t_max=T_MAX, beta=beta, gamma=gamma)
    s_mat, i_mat, r_mat, ml = pad_results(res)
    mean_i = np.nanmean(i_mat, axis=0)
    plt.plot(range(len(mean_i)), mean_i, color=color, label=label, alpha=0.8)

plt.xlabel("Passo temporale $t$")
plt.ylabel("Infetti medi E[I]")
plt.title("Confronto scenari: effetto di β e γ")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Numero Riproduzione di Base (R₀)

Per il modello SIR, il numero riproduttivo di base è:

$$ R_0 = \frac{\beta}{\gamma} $$

Se $R_0 > 1$ l'epidemia si propaga; se $R_0 < 1$ si estingue subito.

Con i nostri parametri: $R_0 = 0.2 / 0.1 = 2.0$

In [ ]:
R0 = BETA / GAMMA
print(f"R₀ = β/γ = {BETA}/{GAMMA} = {R0}")
print("Interpretazione: un infetto genera in media", R0, "nuovi infetti.")
if R0 > 1:
    print("→ R₀ > 1: l'epidemia si propaga (fase epidemica).")
else:
    print("→ R₀ < 1: l'epidemia si estingue spontaneamente.")

---
*Notebook generato per l'esame orale di Modelli Probabilistici.*